# Task 3 (Reproduction): PIDNet-S on CamVid

Reproducing **PIDNet-S** on CamVid (11 semantic classes, `960x720`, random init on CPU).

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# Auto-detect Kaggle environment
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()

if IS_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yacs", "thop"])
    repo_dest = Path("/kaggle/working/AI512_Project")
    if not repo_dest.exists():
        subprocess.run(["git", "clone", "https://github.com/Gitansh-Bansal/AI512_Project.git", str(repo_dest)])
    else:
        subprocess.run(["git", "-C", str(repo_dest), "pull"])
    os.chdir(str(repo_dest))

# Invalidate cached modules so updates apply without kernel restart
for mod_name in list(sys.modules.keys()):
    if any(prefix in mod_name for prefix in ['train_camvid', 'test_camvid', 'datasets.camvid', 'models.pidnet']):
        del sys.modules[mod_name]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

CWD = Path(".").resolve()
REPO_ROOT = next((p for p in [CWD] + list(CWD.parents) if (p / "PIDNet").exists()), CWD)
PIDNET_DIR = REPO_ROOT / "PIDNet"
if str(PIDNET_DIR) not in sys.path:
    sys.path.insert(0, str(PIDNET_DIR))

# Auto-locate CamVid dataset in /kaggle/input or local
DATA_ROOT = None
if Path("/kaggle/input").exists():
    for f in Path("/kaggle/input").rglob("*0001TP*"):
        for p_cand in f.parents:
            if (p_cand / "images").exists() or (p_cand / "train").exists() or (p_cand / "camvid").exists() or (p_cand / "CamVid").exists() or p_cand.parent == Path("/kaggle/input"):
                DATA_ROOT = p_cand
                break
        if DATA_ROOT:
            break
    if DATA_ROOT is None:
        for d in Path("/kaggle/input").glob("*camvid*"):
            if d.is_dir():
                DATA_ROOT = d
                break

if DATA_ROOT is None:
    DATA_ROOT = PIDNET_DIR / "data"

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")

from models.pidnet import get_pred_model
from datasets.camvid import CamVid
from metrics import SegmentationMetrics, CAMVID_CLASSES
from benchmarks import get_hardware_info, profile_parameters, profile_flops, benchmark_inference_speed
from train_camvid import run_training, plot_training_history
from test_camvid import colorize_camvid_mask

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | PyTorch: {torch.__version__}")


## 1. Model, Complexity & Speed

In [ ]:
model = get_pred_model(name="pidnet_s", num_classes=11).to(device).eval()

with torch.no_grad():
    out = model(torch.randn(1, 3, 720, 960, device=device))
print("Input: (1, 3, 720, 960) -> Output:", tuple(out.shape))

param_info = profile_parameters(model)
flop_profiles = profile_flops(model, [(720, 960)], device=device)
hw_info = get_hardware_info(device)
speed = benchmark_inference_speed(model, input_resolution=(720, 960), batch_size=1, device=device, warmup_runs=5, timed_runs=15)

print(f"Params: {param_info['total_params_M']:.2f} M | Hardware: {hw_info['CPU_Processor']}")
print(f"FLOPs (960x720): {flop_profiles[0]['FLOPs_G']:.2f} G | Latency: {speed['Mean_Latency_ms']:.1f} ms | FPS: {speed['FPS']:.2f}")

## 2. Training (CamVid, 8 Epochs)

In [ ]:
TRAIN_CONFIG = {
    "epochs": 10 if torch.cuda.is_available() else 4,
    "batch_size": 4 if torch.cuda.is_available() else 2,
    "optimizer": "adamw",
    "base_lr": 0.001,
    "crop_size": (288, 384),
    "val_freq": 1,
    "val_samples": None if torch.cuda.is_available() else 50,
    "save_dir": str(REPO_ROOT / "output" / "camvid_pidnet_train"),
}

RETRAIN = True if (IS_KAGGLE and torch.cuda.is_available()) else False
history_path = Path(TRAIN_CONFIG["save_dir"]) / "training_history.json"
ckpt_path = Path(TRAIN_CONFIG["save_dir"]) / "best_pidnet_camvid.pt"

if RETRAIN or not (history_path.exists() and ckpt_path.exists()):
    history = run_training(
        data_root=str(DATA_ROOT),
        train_list="list/camvid/train.lst",
        val_list="list/camvid/val.lst",
        epochs=TRAIN_CONFIG["epochs"],
        batch_size=TRAIN_CONFIG["batch_size"],
        optimizer_name=TRAIN_CONFIG["optimizer"],
        base_lr=TRAIN_CONFIG["base_lr"],
        crop_h=TRAIN_CONFIG["crop_size"][0],
        crop_w=TRAIN_CONFIG["crop_size"][1],
        val_freq=TRAIN_CONFIG["val_freq"],
        val_samples=TRAIN_CONFIG["val_samples"],
        save_dir=TRAIN_CONFIG["save_dir"],
    )
else:
    history = json.loads(history_path.read_text())
    print(f"Loaded {len(history)}-epoch history from {history_path.name}")

plot_training_history(history)
plt.show()


## 3. Evaluation (CamVid Test Set, N=233)

In [ ]:
eval_ds = CamVid(
    root=DATA_ROOT, list_path="list/camvid/test.lst",
    num_classes=11, multi_scale=False, flip=False, base_size=960, crop_size=(720, 960),
)
test_loader = DataLoader(eval_ds, batch_size=2, shuffle=False)

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
state = ckpt.get("state_dict", ckpt)
model.load_state_dict({k.replace("module.", "").replace("model.", ""): v for k, v in state.items()}, strict=False)
model.eval()

metrics = SegmentationMetrics(num_classes=11, ignore_label=255, class_names=CAMVID_CLASSES)
mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
vis_samples = []

with torch.no_grad():
    for imgs, lbls, _, _, names in test_loader:
        preds = model(imgs.to(device))
        if isinstance(preds, list):
            preds = preds[1]
        if preds.shape[-2:] != lbls.shape[-2:]:
            preds = F.interpolate(preds, size=lbls.shape[-2:], mode="bilinear", align_corners=False)
        pred_cls = torch.argmax(preds, dim=1).cpu().numpy()
        metrics.update(pred_cls, lbls.numpy())

        for b in range(imgs.shape[0]):
            if len(vis_samples) < 3:
                rgb = np.clip((imgs[b].permute(1, 2, 0).numpy() * std + mean) * 255, 0, 255).astype(np.uint8)
                vis_samples.append((names[b], rgb, lbls[b].numpy(), pred_cls[b]))

summary = metrics.compute_metrics()
print(f"Evaluated {len(eval_ds)} CamVid test images:")
print(f"  mIoU:           {summary['mIoU']*100:.2f}%")
print(f"  Mean Dice (F1): {summary['mean_Dice']*100:.2f}%")
print(f"  Sensitivity:    {summary['mean_Sensitivity']*100:.2f}%")
print(f"  Specificity:    {summary['mean_Specificity']*100:.2f}%")
print(f"  Pixel Accuracy: {summary['Pixel_Accuracy']*100:.2f}%")

## 4. Per-Class Breakdown & Sample Predictions

In [ ]:
df_cls = pd.DataFrame(summary["Per_Class"])
for col in ["IoU", "Dice", "Sensitivity", "Specificity"]:
    df_cls[f"{col} (%)"] = (df_cls.pop(col) * 100).round(2)
display(df_cls[["Class_ID", "Class_Name", "Pixels_GT", "IoU (%)", "Dice (%)", "Sensitivity (%)", "Specificity (%)"]])

fig, axes = plt.subplots(len(vis_samples), 3, figsize=(15, 3.8 * len(vis_samples)))
for i, (name, rgb, gt, pred) in enumerate(vis_samples):
    axes[i, 0].imshow(rgb); axes[i, 0].set_title(name); axes[i, 0].axis("off")
    axes[i, 1].imshow(colorize_camvid_mask(gt)); axes[i, 1].set_title("Ground Truth"); axes[i, 1].axis("off")
    axes[i, 2].imshow(colorize_camvid_mask(pred)); axes[i, 2].set_title("PIDNet-S Prediction"); axes[i, 2].axis("off")
plt.tight_layout()
plt.show()

## 5. Paper vs. Reproduction Comparison

In [ ]:
display(pd.DataFrame({
    "Metric": ["Dataset", "Init & Schedule", "Params", "Speed", "Test mIoU"],
    "PIDNet-S Paper (Xu et al., 2023)": ["CamVid (960x720)", "Cityscapes pretrain, 200 ep (2 GPUs)", "7.60 M", "153.7 FPS (RTX 3090)", "80.10%"],
    "Ours (CPU Scratch)": [
        "CamVid (960x720)",
        f"Random init, {len(history)} ep (CPU)",
        f"{param_info['total_params_M']:.2f} M",
        f"{speed['FPS']:.2f} FPS (CPU)",
        f"{summary['mIoU']*100:.2f}% (Acc: {summary['Pixel_Accuracy']*100:.2f}%)",
    ],
}))